# Long-term Memory RAG v1 — official retrieval evaluation

This notebook runs the user-memory retriever against released **LoCoMo** evidence annotations. It does not require a paid API key. A Colab T4/L4 GPU is recommended for multilingual E5.

Before running: **Runtime → Change runtime type → T4 GPU**.

In [ ]:
!nvidia-smi
!pip -q install sentence-transformers==3.4.1 scikit-learn==1.6.1
!git clone -q https://github.com/nanlalala/cbt-memory-system.git
%cd cbt-memory-system

## Download the official released dataset

In [ ]:
from pathlib import Path
from urllib.request import urlretrieve
data_dir = Path('evaluation/memory-retrieval/data')
data_dir.mkdir(parents=True, exist_ok=True)
urlretrieve('https://raw.githubusercontent.com/snap-research/locomo/3eb6f2c585f5e1699204e3c3bdf7adc5c28cb376/data/locomo10.json', data_dir / 'locomo10.json')
print('Downloaded:', *(p.name for p in data_dir.iterdir()))

## 1. Fast CPU baseline

LoCoMo uses the official dialogue-turn evidence IDs.

In [ ]:
!python evaluation/memory-retrieval/run_memory_retrieval_benchmark.py \
  --dataset locomo --input evaluation/memory-retrieval/data/locomo10.json \
  --backend lexical --output evaluation/memory-retrieval/outputs/locomo_lexical.json

## 2. Multilingual E5 and hybrid retrieval

These runs use the same questions and evidence IDs. Only the retrieval backend changes.

In [ ]:
!python evaluation/memory-retrieval/run_memory_retrieval_benchmark.py \
  --dataset locomo --input evaluation/memory-retrieval/data/locomo10.json \
  --backend dense --device cuda --output evaluation/memory-retrieval/outputs/locomo_e5.json
!python evaluation/memory-retrieval/run_memory_retrieval_benchmark.py \
  --dataset locomo --input evaluation/memory-retrieval/data/locomo10.json \
  --backend hybrid --device cuda --output evaluation/memory-retrieval/outputs/locomo_hybrid.json

## Interpretation

Report Recall@1/5/10 and MRR@10 for each backend. The Week 4 end-to-end comparison will keep the response model and CBT Knowledge RAG fixed while changing only the memory condition.